# 02 · Dedupe threshold sweep

**Question:** at what cosine similarity should two messages count as the same
household (ADR 0003)?

Too low merges neighbors (low precision); too high misses relatives' reports
(low recall). The Graph v2 dev run at 0.65 scored **50.0% / 81.8%** (P / R),
so 0.65 merges too much.

The first threshold pays for the model and embedding calls; every other
threshold reuses them from the cache, so the sweep is nearly free.

In [ ]:
import lab
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
run_settings = lab.settings()
base = lab.build_deps(run_settings)  # one geocoder cache shared across the sweep
THRESHOLDS = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

runs = {}
for threshold in THRESHOLDS:
    runs[threshold] = lab.run(lab.with_dedupe_threshold(base, threshold), run_settings)
    print(threshold, "done")

In [ ]:
sweep = pd.DataFrame(
    {
        "precision": {t: lab.metric(r, "Dedupe precision") for t, r in runs.items()},
        "recall": {t: lab.metric(r, "Dedupe recall") for t, r in runs.items()},
    }
)
sweep["f1"] = 2 * sweep.precision * sweep.recall / (sweep.precision + sweep.recall)
sweep

In [ ]:
ax = sweep[["precision", "recall", "f1"]].plot(marker="o", ylim=(0, 1.05), grid=True)
ax.set_xlabel("cosine similarity threshold")
ax.set_title("Dedupe on dev: merging neighbors vs missing relatives")
plt.show()

## Checks before choosing
- Critical recall must not move: dedupe changes cases, not priorities.
- A wrong merge hides a household behind another; prefer precision when F1 ties.

In [ ]:
lab.compare({f"t={t}": r for t, r in runs.items()}).loc[["Critical recall", "Dedupe P / R"]]

## Decision
Write the chosen threshold and why here, then set `SAKLOLO_DEDUPE_SIMILARITY_THRESHOLD`
in `.env.example` and the default in `config.py`.